# Célula 1 — Markdown
"""
## 3. Methodology

### 3.1 Evaluation Protocol
Para cada amostra do subconjunto, executamos inferência com ambos os modelos
e coletamos: transcrição/tradução gerada, WER, BLEU e latência de inferência.
"""

In [ ]:
# Célula 2 — Funções de métricas
from jiwer import wer as calcular_wer
from sacrebleu.metrics import BLEU
import time

bleu_metric = BLEU(effective_order=True)

def avaliar_amostra(hipotese_transcricao, hipotese_traducao,
                    referencia_transcricao, referencia_traducao,
                    tempo_inferencia, duracao_audio):
    return {
        "WER":      calcular_wer(referencia_transcricao, hipotese_transcricao),
        "BLEU":     bleu_metric.sentence_score(hipotese_traducao, [referencia_traducao]).score,
        "latencia_s":   tempo_inferencia,
        "RTF":      tempo_inferencia / duracao_audio,  # Real-Time Factor — métrica chave
    }

In [ ]:
# Célula 3 — Inferência SeamlessM4T
import torch
import torchaudio

resultados_seamless = []

for amostra in subset.select(range(50)):  # começa com 50 para testar
    audio = torch.tensor(amostra['audio']['array']).unsqueeze(0).float()
    sr    = amostra['audio']['sampling_rate']
    
    inputs = processor_seamless(
        audios=audio, sampling_rate=sr,
        src_lang="por", tgt_lang="eng",
        return_tensors="pt"
    )
    
    t0 = time.perf_counter()
    with torch.no_grad():
        output_tokens = model_seamless.generate(**inputs, tgt_lang="eng")
    latencia = time.perf_counter() - t0
    
    traducao = processor_seamless.decode(output_tokens[0].tolist(), skip_special_tokens=True)
    duracao  = len(amostra['audio']['array']) / sr
    
    metricas = avaliar_amostra(
        hipotese_transcricao=traducao,   # SeamlessM4T faz S2TT direto
        hipotese_traducao=traducao,
        referencia_transcricao=amostra['sentence'],
        referencia_traducao=amostra['translation'],
        tempo_inferencia=latencia,
        duracao_audio=duracao
    )
    metricas['modelo'] = 'SeamlessM4T-v2'
    resultados_seamless.append(metricas)

print(f"SeamlessM4T — {len(resultados_seamless)} amostras processadas")

In [ ]:
# Célula 4 — Inferência NeMo Canary
resultados_canary = []

model_canary.eval()

for amostra in subset.select(range(50)):
    audio_path = f"/tmp/amostra_{amostra['id']}.wav"
    torchaudio.save(audio_path,
                    torch.tensor(amostra['audio']['array']).unsqueeze(0),
                    amostra['audio']['sampling_rate'])
    duracao = len(amostra['audio']['array']) / amostra['audio']['sampling_rate']

    t0 = time.perf_counter()
    resultado = model_canary.transcribe(
        audio=[audio_path],
        batch_size=1,
        task="s2t_translation",
        source_lang="pt",
        target_lang="en"
    )
    latencia = time.perf_counter() - t0

    metricas = avaliar_amostra(
        hipotese_transcricao=resultado[0].text,
        hipotese_traducao=resultado[0].text,
        referencia_transcricao=amostra['sentence'],
        referencia_traducao=amostra['translation'],
        tempo_inferencia=latencia,
        duracao_audio=duracao
    )
    metricas['modelo'] = 'NeMo Canary-1B'
    resultados_canary.append(metricas)

In [ ]:
# Célula 5 — Consolidação e Table II do artigo
df = pd.DataFrame(resultados_seamless + resultados_canary)

tabela_resultados = df.groupby('modelo').agg(
    WER_medio=('WER', 'mean'),
    BLEU_medio=('BLEU', 'mean'),
    Latencia_media_s=('latencia_s', 'mean'),
    RTF_medio=('RTF', 'mean')
).round(4)

print(tabela_resultados)
tabela_resultados.to_csv("results/table2_main_results.csv")
# → Esta saída é a Table II do artigo

In [ ]:
# Célula 6 — Figura 2: comparação visual (Figure 2 do artigo)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metricas_plot = ['WER', 'BLEU', 'RTF']
titulos       = ['WER ↓ (menor = melhor)', 'BLEU ↑ (maior = melhor)', 'RTF ↓ (tempo real)']
cores         = {'SeamlessM4T-v2': 'steelblue', 'NeMo Canary-1B': 'coral'}

for ax, metrica, titulo in zip(axes, metricas_plot, titulos):
    for modelo, grupo in df.groupby('modelo'):
        ax.hist(grupo[metrica], alpha=0.65, label=modelo,
                color=cores[modelo], bins=20, edgecolor='white')
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(metrica)
    ax.legend(fontsize=8)

plt.suptitle("Distribuição de métricas por modelo — CoVoST-2 pt→en", fontsize=12)
plt.tight_layout()
plt.savefig("figures/fig2_metrics_comparison.png", dpi=150, bbox_inches='tight')
plt.show()